# Step 01_02_03 -- Raw Schema DESCRIBE: sc2egset

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_02 -- EDA
**Dataset:** sc2egset
**Question:** What are the column names and types for every `*_raw` object in
the sc2egset DuckDB? This establishes a baseline schema snapshot used by
downstream feature engineering steps.
**Invariants applied:** #6 (reproducibility — SQL inlined in artifact), #9 (step scope: query)
**Step scope:** query
**Type:** Read-only — no DuckDB writes, no new tables, no schema changes

In [1]:
import json

import duckdb

from rts_predict.common.notebook_utils import get_reports_dir, setup_notebook_logging
from rts_predict.games.sc2.config import DB_FILE

logger = setup_notebook_logging()
logger.info("DB_FILE: %s", DB_FILE)

12:58:01 INFO notebook: DB_FILE: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/data/db/db.duckdb


## 1. Connect to DuckDB (read-only)

In [2]:
con = duckdb.connect(str(DB_FILE), read_only=True)
print(f"Connected (read-only): {DB_FILE}")

Connected (read-only): /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/data/db/db.duckdb


## 2. DESCRIBE all *_raw objects

Objects covered:
- Tables: `replays_meta_raw`, `replay_players_raw`, `map_aliases_raw`
- Views: `game_events_raw`, `tracker_events_raw`, `message_events_raw`

In [3]:
RAW_OBJECTS = [
    "replays_meta_raw",
    "replay_players_raw",
    "map_aliases_raw",
    "game_events_raw",
    "tracker_events_raw",
    "message_events_raw",
]

schemas: dict[str, list[dict]] = {}

for obj in RAW_OBJECTS:
    df = con.execute(f"DESCRIBE {obj}").df()
    schemas[obj] = df.to_dict(orient="records")
    print(f"\n=== DESCRIBE {obj} ({len(df)} columns) ===")
    print(df.to_string(index=False))


=== DESCRIBE replays_meta_raw (9 columns) ===
      column_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            column_type null  key default extra
         filename                                                                                                                                                                                                                                                                                                                                                                                       

## 3. Write artifact

In [4]:
artifacts_dir = (
    get_reports_dir("sc2", "sc2egset")
    / "artifacts" / "01_exploration" / "02_eda"
)
artifacts_dir.mkdir(parents=True, exist_ok=True)

artifact_data = {
    "step": "01_02_03",
    "dataset": "sc2egset",
    "schemas": schemas,
}

artifact_path = artifacts_dir / "01_02_03_raw_schema_describe.json"
artifact_path.write_text(json.dumps(artifact_data, indent=2, default=str))
logger.info("Artifact written: %s", artifact_path)

print(f"\nArtifact written: {artifact_path}")
for obj, cols in schemas.items():
    print(f"  {obj}: {len(cols)} columns")

12:58:01 INFO notebook: Artifact written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/02_eda/01_02_03_raw_schema_describe.json



Artifact written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/02_eda/01_02_03_raw_schema_describe.json
  replays_meta_raw: 9 columns
  replay_players_raw: 25 columns
  map_aliases_raw: 4 columns
  game_events_raw: 4 columns
  tracker_events_raw: 4 columns
  message_events_raw: 4 columns


In [5]:
con.close()